# A live FAIR assessment of OSDR datasets

This notebook is an **interactive, standardised FAIR assessment** of the data
available through NASA's Open Science Data Repository (OSDR), built directly on
the public [biodata API](https://visualization.osdr.nasa.gov/biodata/api/).

For every dataset it asks: how **F**indable, **A**ccessible, **I**nteroperable
and **R**eusable is it, judged from the metadata the API actually exposes?

> **How to run live:** set `MAX_DATASETS = None` below and re-run to score the
> entire repository. The version shown here scores a sample so the book builds
> quickly. Outputs are embedded; nothing runs when the book is built.

In [1]:
import time
import requests
import pandas as pd
import plotly.express as px
import plotly.io as pio

# CDN-backed renderer => interactive charts survive in the static book HTML
pio.renderers.default = "notebook_connected"

API = "https://visualization.osdr.nasa.gov/biodata/api/v2"

# Score a sample for a fast book build; set to None to score everything.
MAX_DATASETS = 60

## 1. Find every dataset

The REST interface lists all datasets at `/v2/datasets/`. We start there and
traverse into each dataset's metadata.

In [2]:
resp = requests.get(f"{API}/datasets/", timeout=60)
resp.raise_for_status()
all_ids = list(resp.json().keys())
print(f"OSDR currently exposes {len(all_ids)} datasets via the biodata API.")

ids = all_ids if MAX_DATASETS is None else all_ids[:MAX_DATASETS]
print(f"Scoring {len(ids)} of them in this run.")

OSDR currently exposes 640 datasets via the biodata API.
Scoring 60 of them in this run.


## 2. Define the FAIR rubric

Each principle is broken into concrete, checkable sub-criteria. A criterion
scores **1** if the corresponding metadata is present and non-empty, else **0**.
This is intentionally transparent and reproducible — edit the rubric to match
your community's FAIR maturity model.

Note one structural finding baked in below: the biodata API metadata does **not**
expose an explicit **licence** field, so `R1.1 (licence)` will fail for every
dataset. That is a real, repository-wide reusability gap worth flagging.

In [3]:
def present(meta, key):
    """True if a metadata key exists and is not empty."""
    v = meta.get(key)
    return v not in (None, "", [], {}, "n/a", "N/A")

def fair_criteria(meta):
    """Return {principle: {criterion: 0/1}} for one dataset's metadata."""
    p = lambda k: 1 if present(meta, k) else 0
    return {
        "Findable": {
            "F1 accession/identifier": max(p("accession"), p("identifiers"),
                                           p("study identifier")),
            "F2 rich title+description": 1 if (present(meta, "study title") and
                                               present(meta, "study description")) else 0,
            "F2 organism indexed": p("organism"),
            "F4 public release date": p("study public release date"),
        },
        "Accessible": {
            "A1 REST/API retrievable": 1,                  # every dataset has a REST_URL
            "A1 authoritative source url": p("authoritative source url"),
            "A1 file listing available": 1,                # every dataset exposes /files/
        },
        "Interoperable": {
            "I1 assay technology type": p("study assay technology type"),
            "I1 measurement type": p("study assay measurement type"),
            "I2 ISA characteristics/factors": max(p("characteristics"),
                                                  p("study factor type")),
            "I3 data source linkage": max(p("data source accession"),
                                          p("project link")),
        },
        "Reusable": {
            "R1 protocols described": p("study protocol description"),
            "R1.1 explicit licence": 0,                    # not exposed by the API
            "R1.2 provenance (funding/grant)": max(p("study funding agency"),
                                                   p("study grant number")),
            "R1.2 linked publication": max(p("study publication title"),
                                           p("study publication author list")),
        },
    }

## 3. Pull metadata and score every dataset

We fetch each dataset's metadata block and apply the rubric. Failures are
recorded rather than aborting the run, so a single flaky request can't break the
whole assessment.

In [4]:
rows = []
for i, acc in enumerate(ids, 1):
    try:
        r = requests.get(f"{API}/dataset/{acc}/", timeout=60)
        r.raise_for_status()
        meta = r.json()[acc]["metadata"]
    except Exception as e:                      # noqa: BLE001 - keep the sweep going
        rows.append({"accession": acc, "error": str(e)})
        continue

    crit = fair_criteria(meta)
    row = {"accession": acc,
           "organism": meta.get("organism", ""),
           "assay": meta.get("study assay technology type", "")}
    for principle, checks in crit.items():
        row[principle] = round(100 * sum(checks.values()) / len(checks), 1)
        for name, val in checks.items():
            row[name] = val
    row["FAIR overall"] = round(
        sum(row[p] for p in ["Findable", "Accessible", "Interoperable", "Reusable"]) / 4, 1)
    rows.append(row)
    if i % 20 == 0:
        print(f"  scored {i}/{len(ids)}")
    time.sleep(0.05)                            # be polite to the API

df = pd.DataFrame(rows)
ok = df[~df.get("error", pd.Series(index=df.index)).notna()] if "error" in df else df
print(f"Scored {len(ok)} datasets successfully.")
ok.head()

  scored 20/60


  scored 40/60


  scored 60/60
Scored 60 datasets successfully.


,accession,organism,assay,Findable,F1 accession/identifier,F2 rich title+description,F2 organism indexed,F4 public release date,Accessible,A1 REST/API retrievable,...,I1 assay technology type,I1 measurement type,I2 ISA characteristics/factors,I3 data source linkage,Reusable,R1 protocols described,R1.1 explicit licence,R1.2 provenance (funding/grant),R1.2 linked publication,FAIR overall
0,OSD-1,Drosophila melanogaster,DNA microarray,100.0,1,1,1,1,100.0,1,...,1,1,1,1,75.0,1,0,1,1,93.8
1,OSD-2,Homo sapiens,DNA microarray,100.0,1,1,1,1,100.0,1,...,1,1,1,1,75.0,1,0,1,1,93.8
2,OSD-3,Drosophila melanogaster,DNA microarray,100.0,1,1,1,1,100.0,1,...,1,1,1,1,75.0,1,0,1,1,93.8
3,OSD-4,Mus musculus,DNA microarray,100.0,1,1,1,1,100.0,1,...,1,1,1,1,75.0,1,0,1,1,93.8
4,OSD-5,Homo sapiens,DNA microarray,100.0,1,1,1,1,100.0,1,...,1,1,1,1,75.0,1,0,1,1,93.8


## 4. The FAIR scorecard

Per-principle and overall scores (0–100). Sortable in the live notebook; this
is the backbone other views build on.

In [5]:
cols = ["accession", "organism", "Findable", "Accessible",
        "Interoperable", "Reusable", "FAIR overall"]
scorecard = ok[cols].sort_values("FAIR overall", ascending=False).reset_index(drop=True)
scorecard.head(20)

,accession,organism,Findable,Accessible,Interoperable,Reusable,FAIR overall
0,OSD-1,Drosophila melanogaster,100.0,100.0,100.0,75.0,93.8
1,OSD-2,Homo sapiens,100.0,100.0,100.0,75.0,93.8
2,OSD-3,Drosophila melanogaster,100.0,100.0,100.0,75.0,93.8
3,OSD-4,Mus musculus,100.0,100.0,100.0,75.0,93.8
4,OSD-5,Homo sapiens,100.0,100.0,100.0,75.0,93.8
5,OSD-6,Rattus norvegicus,100.0,100.0,100.0,75.0,93.8
6,OSD-7,Arabidopsis thaliana,100.0,100.0,100.0,75.0,93.8
7,OSD-8,Arabidopsis thaliana,100.0,100.0,100.0,75.0,93.8
8,OSD-9,Homo sapiens,100.0,100.0,100.0,75.0,93.8
9,OSD-11,Salmonella enterica,100.0,100.0,100.0,75.0,93.8


## 5. Where does OSDR do well — and where are the gaps?

Average score per FAIR principle across the sampled datasets.

In [6]:
means = ok[["Findable", "Accessible", "Interoperable", "Reusable"]].mean().round(1)
fig = px.bar(means, orientation="h",
             labels={"value": "Average score (0-100)", "index": "FAIR principle"},
             title="Average FAIR principle scores across sampled OSDR datasets",
             text=means.values)
fig.update_layout(showlegend=False, height=350)
fig.show()

In [7]:
# Which individual sub-criteria are most often missing?
subcrit = [c for c in ok.columns if c[:2] in ("F1","F2","F4","A1","I1","I2","I3","R1")]
gap = (100 * (1 - ok[subcrit].mean())).round(1).sort_values(ascending=False)
fig2 = px.bar(gap, orientation="h",
              labels={"value": "% of datasets missing this", "index": "FAIR criterion"},
              title="Most common FAIR metadata gaps in OSDR")
fig2.update_layout(showlegend=False, height=500)
fig2.show()

In [8]:
# Distribution of overall FAIR scores
fig3 = px.histogram(ok, x="FAIR overall", nbins=20,
                    title="Distribution of overall FAIR scores")
fig3.update_layout(height=350)
fig3.show()

## 7. How much of OSDR is plant data?

Before zooming into the plant studies, let's see where plants sit within the
*whole* repository — computed **live from the OSDR API** (so it matches the
*Interactive OSDR API explorer* chapter). We pull every sample's organism in one
query, collapse to one organism per dataset, and measure the **plant share**.

In [9]:
# Live: pull every sample's organism in one query, collapse to one per dataset
import io
q = f"{API}/query/metadata/?id.accession&study.characteristics.organism&format=csv"
org_tbl = pd.read_csv(io.StringIO(requests.get(q, timeout=120).text))
per_dataset = org_tbl.groupby("id.accession")["study.characteristics.organism"].first()

PLANT = "arabidopsis|thaliana|brassica|oryza|plant"
is_plant = per_dataset.str.contains(PLANT, case=False, na=False)
plant_ids = per_dataset[is_plant].index.tolist()
n_plant, n_total = int(is_plant.sum()), len(per_dataset)
print(f"{n_plant} of {n_total} OSDR datasets are plants "
      f"({100 * n_plant / n_total:.1f}%) — live from the API")

70 of 640 OSDR datasets are plants (10.9%) — live from the API


In [10]:
# Plant share of the whole repository
fig = px.pie(names=["Plants", "All other organisms"],
             values=[n_plant, n_total - n_plant], hole=0.55,
             color=["Plants", "All other organisms"],
             color_discrete_map={"Plants": "#2e7d32", "All other organisms": "#cfd8dc"},
             title=f"Plants make up {100 * n_plant / n_total:.0f}% of OSDR datasets (live)")
fig.update_layout(height=380)
fig.show()

In [11]:
# Where Arabidopsis ranks among all OSDR organisms (live)
top = per_dataset.value_counts().head(12).rename_axis("organism").reset_index(name="datasets")
top["plant"] = top["organism"].str.contains(PLANT, case=False, na=False)
fig = px.bar(top.sort_values("datasets"), x="datasets", y="organism", orientation="h",
             color="plant", color_discrete_map={True: "#2e7d32", False: "#90a4ae"},
             title="Top organisms in OSDR (live; plants in green)")
fig.update_layout(height=420, showlegend=False)
fig.show()

### Zoom in: the plant subset at higher resolution

Now we fetch each plant dataset's full metadata once (live) and characterise the
subset by **species**, **assay technology**, **experimental factors**, and **time** —
and reuse the very same records to score their FAIR-ness in the next section.

In [12]:
# One live metadata sweep over the plant datasets -> powers the drill-down AND the FAIR scores
import datetime

def as_list(v):
    if isinstance(v, list):
        return [str(x) for x in v if str(x).strip()]
    return [str(v)] if v not in (None, "") else []

plant_rows = []
for acc in plant_ids:
    try:
        meta = requests.get(f"{API}/dataset/{acc}/", timeout=60).json()[acc]["metadata"]
    except Exception:
        continue
    org = meta.get("organism", "")
    org = org[0] if isinstance(org, list) and org else (org if isinstance(org, str) else "")
    try:
        year = datetime.datetime.utcfromtimestamp(int(meta.get("study public release date"))).year
    except Exception:
        year = None
    crit = fair_criteria(meta)
    row = {"accession": acc, "organism": org, "year": year,
           "assay": as_list(meta.get("study assay technology type")),
           "factors": as_list(meta.get("study factor type"))}
    for principle, checks in crit.items():
        row[principle] = round(100 * sum(checks.values()) / len(checks), 1)
    row["FAIR overall"] = round(
        sum(row[p] for p in ["Findable", "Accessible", "Interoperable", "Reusable"]) / 4, 1)
    plant_rows.append(row)
    time.sleep(0.03)

plant_df = pd.DataFrame(plant_rows)
print(f"Plant subset (live): {len(plant_df)} datasets characterised")

C:\Users\drric\AppData\Local\Temp\ipykernel_37524\1153958008.py:18: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  year = datetime.datetime.utcfromtimestamp(int(meta.get("study public release date"))).year


Plant subset (live): 70 datasets characterised


In [13]:
# (a) Species
sp = plant_df["organism"].value_counts().sort_values()
fig = px.bar(sp, orientation="h", text=sp.values,
             labels={"value": "datasets", "index": "species"}, title="Plant species in OSDR")
fig.update_layout(height=300, showlegend=False); fig.show()

In [14]:
# (b) Assay technologies in the plant subset
at = plant_df.explode("assay")["assay"].dropna().value_counts().sort_values()
fig = px.bar(at, orientation="h", text=at.values,
             labels={"value": "datasets", "index": "technology"},
             title="Assay technologies in the plant subset")
fig.update_layout(height=380, showlegend=False); fig.show()

In [15]:
# (c) Experimental factors studied in plant datasets
fac = plant_df.explode("factors")["factors"].dropna()
fac = fac[fac.str.strip() != ""].value_counts().head(12).sort_values()
fig = px.bar(fac, orientation="h", text=fac.values,
             labels={"value": "datasets", "index": "factor"},
             title="Experimental factors in plant studies (e.g. spaceflight, radiation)")
fig.update_layout(height=420, showlegend=False); fig.show()

In [16]:
# (d) Plant datasets over time
yr = plant_df["year"].dropna().astype(int).value_counts().sort_index()
fig = px.bar(x=yr.index, y=yr.values, text=yr.values,
             labels={"x": "Year", "y": "plant datasets"},
             title="Plant datasets released in OSDR by year")
fig.update_layout(height=340); fig.show()

## 8. Astrobotany view — FAIR scores for the plant subset

The same live records, now scored against the FAIR rubric from section 2.

In [17]:
# Average FAIR scores across the plant datasets (live)
pm = plant_df[["Findable", "Accessible", "Interoperable", "Reusable"]].mean().round(1)
fig = px.bar(pm, orientation="h", text=pm.values,
             labels={"value": "Average score (0-100)", "index": "FAIR principle"},
             title="Average FAIR scores across plant (Arabidopsis) OSDR datasets")
fig.update_layout(height=350, showlegend=False); fig.show()

In [18]:
# Spotlight the CARA hero dataset used throughout this book
plant_df.sort_values("FAIR overall", ascending=False).head(15)

,accession,organism,year,assay,factors,Findable,Accessible,Interoperable,Reusable,FAIR overall
1,OSD-121,Arabidopsis thaliana,2017.0,"[DNA microarray, Image analysis, ELISA]",[Space Flight],100.0,100.0,100.0,75.0,93.8
2,OSD-134,Arabidopsis thaliana,2017.0,[DNA microarray],"[Atmospheric Pressure, Organism Part]",100.0,100.0,100.0,75.0,93.8
4,OSD-144,Arabidopsis thaliana,2014.0,[DNA microarray],"[cell cycle phase, Weightlessness Simulation]",100.0,100.0,100.0,75.0,93.8
3,OSD-136,Arabidopsis thaliana,2017.0,[DNA microarray],[Atmospheric Pressure],100.0,100.0,100.0,75.0,93.8
5,OSD-147,Arabidopsis thaliana,2017.0,[DNA microarray],"[Space Flight, genotype]",100.0,100.0,100.0,75.0,93.8
6,OSD-16,Arabidopsis thaliana,1970.0,[mass spectrometry],"[Space Flight, organism part]",100.0,100.0,100.0,75.0,93.8
67,OSD-8,Arabidopsis thaliana,1970.0,[DNA microarray],"[Gravity, Altered, Magnetic Field]",100.0,100.0,100.0,75.0,93.8
7,OSD-17,Arabidopsis thaliana,2012.0,[DNA microarray],"[Space Flight, organism part]",100.0,100.0,100.0,75.0,93.8
8,OSD-193,Arabidopsis thaliana,2018.0,[RNA Sequencing (RNA-Seq)],"[time, genotype, Space Flight]",100.0,100.0,100.0,75.0,93.8
9,OSD-205,Arabidopsis thaliana,2018.0,[DNA microarray],"[Space Flight, Genotype]",100.0,100.0,100.0,75.0,93.8


In [19]:
plant_df[plant_df["accession"] == "OSD-120"]

,accession,organism,year,assay,factors,Findable,Accessible,Interoperable,Reusable,FAIR overall
0,OSD-120,Arabidopsis thaliana,2017.0,"[RNA Sequencing (RNA-Seq), Photography]","[Ecotype, Space Flight, treatment]",100.0,100.0,75.0,75.0,87.5


## 9. Next steps for the publication tool

This notebook is the **FAIR backbone**. Natural extensions:

- **Drill-down explorer** — pick any accession and render its assays, samples
  and data columns via `/v2/dataset/{acc}/assays/` and `/v2/query/data/`.
- **Data storytelling** — join FAIR scores with `study factor type` / `organism`
  to narrate *what* science is well-described vs. under-documented.
- **Submitter feedback** — turn the gap chart into per-dataset, actionable
  "to improve reusability, add X" reports.
- **Refresh on schedule** — run with `MAX_DATASETS = None` to score the whole
  repository and track FAIR maturity over time.